# Bulk Upload Tool

Uploads videos from a local folder directly to R2 and creates Supabase records — bypassing the browser upload UI for much faster bulk ingestion.

### Folder structure

```
VIDEOS TO UPLOAD/
  Bachata/                   → tags: [Bachata]
  Salsa/                     → tags: [Salsa]
  Bachata, Salsa/            → tags: [Bachata, Salsa]  (comma-separated)
```

- Folder name must **exactly** match tag names in the database (case-sensitive)
- Comma-separated folder names apply multiple tags to every video inside
- If **any** tag is not found in the DB, the entire run aborts before touching anything

### What it does

1. **Validate** all subfolder names against DB tags — abort if any are missing
2. **Scan** all video files and check for duplicates already in the DB
3. **Preview** exactly what will be uploaded (dry run by default)
4. **Upload** video to R2 → generate thumbnail via ffmpeg → upload thumbnail to R2
5. **Create** Supabase media record with tags, duration, `recorded_at`, metadata

### `recorded_at` strategy (priority order)

1. `com.apple.quicktime.creationdate` atom — iOS ground truth (most reliable for iPhone videos)
2. `©day` atom — Android, GoPro, QuickTime cameras
3. `mvhd.creation_time` — MP4 header timestamp (skipped if 0, which iOS often sets)
4. Filename datetime pattern — e.g. `VID_20250315_143022.mp4`
5. `st_ctime` — Windows file creation time (last resort; accurate only if file was transferred once)
6. null

### Dependencies

```
pip install imageio-ffmpeg supabase boto3 python-dotenv tabulate
```

In [1]:
import os
from dotenv import load_dotenv

# ══════════════════════════════════════════════
# CONFIG — edit these as needed
# ══════════════════════════════════════════════

# Set to True to preview without uploading anything
DRY_RUN = True

# Root folder containing subfolders named after tags
UPLOAD_ROOT = r"C:\Users\minds\Videos\DANCE\VIDEOS TO UPLOAD"

# Video file extensions to process
VIDEO_EXTENSIONS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}

# ══════════════════════════════════════════════

load_dotenv()

def clean_env(key):
    val = os.getenv(key, "")
    return val.split("#")[0].strip() if val else ""

SUPABASE_URL = clean_env("VITE_SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = clean_env("VITE_SUPABASE_SERVICE_ROLE_KEY")
R2_ACCOUNT_ID = clean_env("R2_ACCOUNT_ID")
R2_ACCESS_KEY_ID = clean_env("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = clean_env("R2_SECRET_ACCESS_KEY")
R2_BUCKET_NAME = clean_env("R2_BUCKET_NAME")

missing_cfg = [k for k, v in {
    "SUPABASE_URL": SUPABASE_URL,
    "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY,
    "R2_ACCOUNT_ID": R2_ACCOUNT_ID,
    "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
    "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
    "R2_BUCKET_NAME": R2_BUCKET_NAME,
}.items() if not v]

if missing_cfg:
    raise RuntimeError(f"Missing config: {', '.join(missing_cfg)}")

mode = "DRY RUN (no changes will be made)" if DRY_RUN else "LIVE RUN (will upload & create DB records)"
print(f"Config loaded. Mode: {mode}")
print(f"Upload root: {UPLOAD_ROOT}")

Config loaded. Mode: DRY RUN (no changes will be made)
Upload root: C:\Users\minds\Videos\DANCE\VIDEOS TO UPLOAD


In [2]:
import boto3
from supabase import create_client

sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
print("Supabase connected.")

r2 = boto3.client(
    "s3",
    endpoint_url=f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name="auto",
)
print("R2 connected.")

Supabase connected.
R2 connected.


---
## Step 1: Validate Folder Names Against DB Tags

Aborts immediately if any subfolder name doesn't match a tag in the database.

In [3]:
from tabulate import tabulate

# ── Load all tags from DB ──
all_tags_res = sb.table("tags").select("id, name").execute()
tag_name_to_id = {t["name"]: t["id"] for t in all_tags_res.data}
print(f"Loaded {len(tag_name_to_id)} tags from DB.")

# ── Scan subfolders ──
if not os.path.isdir(UPLOAD_ROOT):
    raise RuntimeError(f"Upload root not found: {UPLOAD_ROOT}")

subfolders = [
    f for f in os.listdir(UPLOAD_ROOT)
    if os.path.isdir(os.path.join(UPLOAD_ROOT, f)) and not f.startswith(".")
]

if not subfolders:
    raise RuntimeError(f"No subfolders found in {UPLOAD_ROOT}")

print(f"Found {len(subfolders)} subfolder(s): {subfolders}")
print()

# ── Validate each folder name against DB tags ──
# folder_tag_map: folder_name -> list of tag_ids
folder_tag_map = {}   # folder name -> [tag_id, ...]
validation_errors = []

for folder in subfolders:
    tag_names = [t.strip() for t in folder.split(",")]
    tag_ids = []
    for tag_name in tag_names:
        if tag_name in tag_name_to_id:
            tag_ids.append(tag_name_to_id[tag_name])
        else:
            validation_errors.append(f"Tag '{tag_name}' (from folder '{folder}') not found in DB")
    if tag_ids:
        folder_tag_map[folder] = tag_ids

# ── Show validation results ──
rows = []
for folder in subfolders:
    tag_names = [t.strip() for t in folder.split(",")]
    resolved = []
    for t in tag_names:
        resolved.append(f"{t} ({'OK' if t in tag_name_to_id else 'NOT FOUND'})")
    rows.append([folder, ", ".join(resolved)])

print(tabulate(rows, headers=["Folder", "Tags"], tablefmt="simple"))

if validation_errors:
    print()
    print("ABORT — fix these errors before proceeding:")
    for err in validation_errors:
        print(f"  ✗ {err}")
    raise RuntimeError("Tag validation failed. Fix folder names to match DB tags exactly.")

print()
print("All folder names validated successfully.")

Loaded 9 tags from DB.
Found 7 subfolder(s): ['Bachata', 'Cha Cha Cha', 'Kizomba', 'On1,Salsa', 'On2,Salsa', 'Salsa,Temp', 'Tango']

Folder       Tags
-----------  ---------------------
Bachata      Bachata (OK)
Cha Cha Cha  Cha Cha Cha (OK)
Kizomba      Kizomba (OK)
On1,Salsa    On1 (OK), Salsa (OK)
On2,Salsa    On2 (OK), Salsa (OK)
Salsa,Temp   Salsa (OK), Temp (OK)
Tango        Tango (OK)

All folder names validated successfully.


---
## Step 2: Scan Files & Check for Duplicates

In [4]:
import re
import struct
from datetime import datetime, timezone, timedelta
from pathlib import Path

DATETIME_RE = re.compile(r'(\d{4})[-_](\d{2})[-_](\d{2})[-_](\d{2})[-_](\d{2})[-_](\d{2})')
MP4_EPOCH = datetime(1904, 1, 1, tzinfo=timezone.utc)


# ── Atom-based date extraction helpers ──────────────────────────────────────

def _is_valid_date(text: str) -> bool:
    if not text or len(text) < 10:
        return False
    try:
        d = datetime.fromisoformat(text.replace('Z', '+00:00'))
        return 2000 <= d.year <= 2035
    except ValueError:
        return False


def _normalize_date(text: str) -> str:
    """Append +00:00 if no timezone present."""
    t = text.strip()
    if t.endswith('Z') or (len(t) > 19 and ('+' in t[19:] or '-' in t[19:])):
        return t
    return t + '+00:00'


def _parse_apple_creation_date(data: bytes) -> str | None:
    """Read com.apple.quicktime.creationdate UTF-8 value (iOS ground truth)."""
    key = b'com.apple.quicktime.creationdate'
    idx = data.find(key)
    if idx == -1:
        return None
    # Value appears shortly after the key in the ilst atom — scan forward 4KB
    snippet = data[idx: idx + 4096].decode('utf-8', errors='replace')
    # Match e.g. 2025-03-08T14:03:35+0500 or ...Z or ...-05:00
    m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}[+\-Z][^\x00-\x1f]{0,6})', snippet)
    if not m:
        m = re.search(r'(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})', snippet)
    if m and _is_valid_date(m.group(1)):
        val = m.group(1)
        return val if ('+' in val or 'Z' in val or (len(val) > 19 and '-' in val[19:])) else val + '+00:00'
    return None


def _parse_cday_atom(data: bytes) -> str | None:
    """Read ©day (0xA9 64 61 79) — Android, GoPro, QuickTime."""
    tag = b'\xa9day'
    idx = data.find(tag)
    if idx == -1:
        return None

    # Format A — QuickTime udta: [4B tag][2B str_len][2B language][UTF-8]
    if idx + 8 < len(data):
        str_len = struct.unpack('>H', data[idx + 4: idx + 6])[0]
        if 0 < str_len < 256 and idx + 8 + str_len <= len(data):
            text = data[idx + 8: idx + 8 + str_len].decode('utf-8', errors='replace').strip()
            if _is_valid_date(text):
                return _normalize_date(text)

    # Format B — iTunes meta: look for 'data' sub-atom nearby
    data_idx = data.find(b'data', idx + 4)
    if data_idx != -1 and data_idx < idx + 256:
        val_start = data_idx + 4 + 4 + 4  # skip 'data' + type_flag + locale
        val_end = min(val_start + 256, len(data))
        text = data[val_start:val_end].split(b'\x00')[0].decode('utf-8', errors='replace').strip()
        if _is_valid_date(text):
            return _normalize_date(text)

    return None


def _parse_mvhd_creation_time(data: bytes) -> str | None:
    """Extract creation_time from mvhd. Skips if 0 (iOS). Valid year 2000-2035."""
    idx = data.find(b'mvhd')
    if idx == -1:
        return None
    offset = idx + 4
    if offset + 4 > len(data):
        return None
    version = data[offset]
    offset += 4  # skip version + flags

    if version == 0:
        if offset + 8 > len(data):
            return None
        creation_secs = struct.unpack('>I', data[offset: offset + 4])[0]
    elif version == 1:
        if offset + 16 > len(data):
            return None
        hi, lo = struct.unpack('>II', data[offset: offset + 8])
        creation_secs = hi * (2 ** 32) + lo
    else:
        return None

    if creation_secs == 0:
        return None

    try:
        dt = MP4_EPOCH + timedelta(seconds=creation_secs)
        if not (2000 <= dt.year <= 2035):
            return None
        return dt.strftime('%Y-%m-%dT%H:%M:%S+00:00')
    except (OverflowError, OSError):
        return None


def extract_recorded_at(filepath: str) -> str | None:
    """
    Priority cascade:
    1. com.apple.quicktime.creationdate atom (iOS ground truth)
    2. ©day atom (Android, GoPro, QuickTime)
    3. mvhd.creation_time (non-iOS; skipped if 0 or out of range)
    4. Filename datetime pattern (YYYY_MM_DD_HH_MM_SS)
    5. st_ctime (Windows file creation time — last resort)
    6. null
    """
    # Read first 512KB — covers moov atom for most camera files
    try:
        with open(filepath, 'rb') as f:
            data = f.read(512 * 1024)
    except OSError:
        return None

    result = _parse_apple_creation_date(data)
    if result:
        return result

    result = _parse_cday_atom(data)
    if result:
        return result

    result = _parse_mvhd_creation_time(data)
    if result:
        return result

    # Filename datetime pattern
    filename = os.path.basename(filepath)
    m = DATETIME_RE.search(filename)
    if m:
        y, mo, d, h, mi, s = m.groups()
        candidate = f"{y}-{mo}-{d}T{h}:{mi}:{s}+00:00"
        if _is_valid_date(candidate):
            return candidate

    # st_ctime — Windows-only reliable (accurate when file transferred directly from device)
    try:
        ctime = os.stat(filepath).st_ctime
        dt = datetime.fromtimestamp(ctime, tz=timezone.utc)
        if 2000 <= dt.year <= 2035:
            return dt.strftime('%Y-%m-%dT%H:%M:%S+00:00')
    except OSError:
        pass

    return None


# ── Scan all video files across subfolders ──
candidates = []  # list of dicts

for folder in subfolders:
    folder_path = os.path.join(UPLOAD_ROOT, folder)
    tag_ids = folder_tag_map[folder]
    tag_names = [t.strip() for t in folder.split(",")]

    for fname in sorted(os.listdir(folder_path)):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VIDEO_EXTENSIONS:
            continue
        fpath = os.path.join(folder_path, fname)
        fsize = os.path.getsize(fpath)
        candidates.append({
            "filename": fname,
            "filepath": fpath,
            "folder": folder,
            "tag_ids": tag_ids,
            "tag_names": tag_names,
            "file_size_bytes": fsize,
            "ext": ext.lstrip("."),
            "recorded_at": extract_recorded_at(fpath),
        })

print(f"Found {len(candidates)} video file(s) to process.")

# ── Fetch existing records for duplicate check ──
# Match on original_filename + file_size_bytes (same as dedup_audit)
existing = []
offset = 0
while True:
    res = sb.table("media").select(
        "id, original_filename, file_size_bytes"
    ).range(offset, offset + 999).execute()
    existing.extend(res.data)
    if len(res.data) < 1000:
        break
    offset += 1000

# Build a set of (filename_lower, file_size) tuples for fast lookup
existing_keys = {
    (r["original_filename"].lower(), r["file_size_bytes"])
    for r in existing
    if r.get("original_filename") and r.get("file_size_bytes")
}
print(f"Loaded {len(existing)} existing DB records for duplicate check.")

# ── Categorize candidates ──
to_upload = []
duplicates = []

for c in candidates:
    key = (c["filename"].lower(), c["file_size_bytes"])
    if key in existing_keys:
        duplicates.append(c)
    else:
        to_upload.append(c)

print()
print(f"  To upload:  {len(to_upload)}")
print(f"  Duplicates (already in DB, will skip): {len(duplicates)}")

if duplicates:
    print()
    print("Skipping duplicates:")
    for c in duplicates:
        print(f"  {c['filename']} ({c['file_size_bytes'] / 1024 / 1024:.1f} MB) — already in DB")

Found 351 video file(s) to process.
Loaded 0 existing DB records for duplicate check.

  To upload:  351
  Duplicates (already in DB, will skip): 0


---
## Step 3: Build Upload Plan

Pre-computes every UUID, storage path, thumbnail path, duration, and `recorded_at` for all files **before** any uploading happens. Review the full dataframe — if anything looks wrong, stop here before running Step 4.

In [5]:
import uuid as uuid_lib
import mimetypes
import struct
import subprocess
import imageio_ffmpeg
import pandas as pd

# Show every row and column — no truncation
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()


def _parse_mvhd_duration(data: bytes) -> int | None:
    """Parse mvhd atom from a bytes buffer. Returns rounded seconds or None."""
    idx = data.find(b'mvhd')
    if idx == -1:
        return None
    offset = idx + 4
    if offset + 4 > len(data):
        return None
    version = data[offset]
    offset += 4  # skip version + flags
    if version == 0:
        if offset + 16 > len(data):
            return None
        offset += 8  # skip creation + modification time
        timescale = struct.unpack('>I', data[offset:offset + 4])[0]
        offset += 4
        duration = struct.unpack('>I', data[offset:offset + 4])[0]
    elif version == 1:
        if offset + 28 > len(data):
            return None
        offset += 16  # skip creation + modification time
        timescale = struct.unpack('>I', data[offset:offset + 4])[0]
        offset += 4
        duration = struct.unpack('>Q', data[offset:offset + 8])[0]
    else:
        return None
    if timescale == 0 or duration == 0:
        return None
    secs = duration / timescale
    if secs <= 0 or secs > 86400:
        return None
    return round(secs)


def _duration_via_ffprobe(filepath: str) -> int | None:
    """Ask ffprobe for duration. Used as final fallback."""
    try:
        result = subprocess.run(
            [
                FFMPEG_PATH.replace("ffmpeg", "ffprobe"),
                "-v", "quiet",
                "-print_format", "json",
                "-show_entries", "format=duration",
                filepath,
            ],
            capture_output=True, timeout=30,
        )
        if result.returncode == 0:
            import json
            info = json.loads(result.stdout)
            secs = float(info.get("format", {}).get("duration", 0))
            if 0 < secs <= 86400:
                return round(secs)
    except Exception:
        pass

    # ffprobe may live alongside ffmpeg binary — try the exact path variant
    try:
        ffprobe_path = FFMPEG_PATH.replace("ffmpeg-win-x86_64", "ffprobe-win-x86_64")
        if not os.path.exists(ffprobe_path):
            # Try same dir, different name
            import shutil
            ffprobe_path = shutil.which("ffprobe") or ""
        if ffprobe_path and os.path.exists(ffprobe_path):
            result = subprocess.run(
                [
                    ffprobe_path,
                    "-v", "quiet",
                    "-print_format", "json",
                    "-show_entries", "format=duration",
                    filepath,
                ],
                capture_output=True, timeout=30,
            )
            if result.returncode == 0:
                import json
                info = json.loads(result.stdout)
                secs = float(info.get("format", {}).get("duration", 0))
                if 0 < secs <= 86400:
                    return round(secs)
    except Exception:
        pass

    # Last resort: use ffmpeg itself (it prints duration to stderr)
    try:
        result = subprocess.run(
            [FFMPEG_PATH, "-i", filepath],
            capture_output=True, timeout=30,
        )
        import re
        m = re.search(r'Duration:\s*(\d+):(\d+):(\d+\.\d+)', result.stderr.decode("utf-8", errors="replace"))
        if m:
            h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
            secs = h * 3600 + mn * 60 + s
            if 0 < secs <= 86400:
                return round(secs)
    except Exception:
        pass

    return None


def get_duration_from_file(filepath: str) -> int | None:
    """
    Extract duration in whole seconds from a local video file.

    Strategy (in order):
    1. Read first 512KB — covers moov-at-start (faststart) files
    2. Read last 512KB — covers moov-at-end (default iPhone recording layout)
    3. ffmpeg stderr parse — catches any remaining edge cases
    """
    file_size = os.path.getsize(filepath)
    CHUNK = 512 * 1024

    # ── 1. Try start of file ──
    try:
        with open(filepath, "rb") as f:
            head = f.read(CHUNK)
        result = _parse_mvhd_duration(head)
        if result:
            return result
    except OSError:
        return None

    # ── 2. Try end of file (moov-at-end — the common iPhone layout) ──
    if file_size > CHUNK:
        try:
            with open(filepath, "rb") as f:
                f.seek(max(0, file_size - CHUNK))
                tail = f.read(CHUNK)
            result = _parse_mvhd_duration(tail)
            if result:
                return result
        except OSError:
            pass

    # ── 3. ffmpeg fallback ──
    return _duration_via_ffprobe(filepath)


def fmt_duration(seconds):
    if seconds is None:
        return None
    s = int(seconds)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m}:{sec:02d}"


# ── Build the plan — pre-compute everything before touching R2 ──
print(f"Building plan for {len(to_upload)} files (reading duration from each)...")

plan = []
for i, c in enumerate(to_upload):
    file_uuid = str(uuid_lib.uuid4())
    ext = c["ext"]
    media_key = f"videos/{file_uuid}.{ext}"
    thumb_key = f"thumbs/{file_uuid}.webp"
    mime_type = mimetypes.guess_type(c["filename"])[0] or "video/mp4"
    duration_secs = get_duration_from_file(c["filepath"])

    plan.append({
        # visible columns
        "original_filename": c["filename"],
        "uuid":              file_uuid,
        "video_path":        media_key,
        "thumbnail_path":    thumb_key,
        "file_size_mb":      round(c["file_size_bytes"] / 1024 / 1024, 1),
        "duration":          fmt_duration(duration_secs),
        "recorded_at":       c["recorded_at"],
        "tags":              ", ".join(c["tag_names"]),
        "mime_type":         mime_type,
        # private — used by upload cell
        "_filepath":         c["filepath"],
        "_file_size_bytes":  c["file_size_bytes"],
        "_duration_secs":    duration_secs,
        "_tag_ids":          c["tag_ids"],
    })
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(to_upload)}...")

total_gb = sum(p["_file_size_bytes"] for p in plan) / 1024 / 1024 / 1024
no_duration    = sum(1 for p in plan if p["_duration_secs"] is None)
no_recorded_at = sum(1 for p in plan if p["recorded_at"] is None)

print(f"\nPlan ready — {len(plan)} files, {total_gb:.2f} GB total")
print(f"  Missing duration:    {no_duration}")
print(f"  Missing recorded_at: {no_recorded_at}")
print()

# ── Display full dataframe ──
display_cols = [
    "original_filename", "uuid", "video_path", "thumbnail_path",
    "file_size_mb", "duration", "recorded_at", "tags", "mime_type",
]
df_plan = pd.DataFrame(plan)[display_cols]
display(df_plan)

Building plan for 351 files (reading duration from each)...
  50/351...
  100/351...
  150/351...
  200/351...
  250/351...
  300/351...
  350/351...

Plan ready — 351 files, 38.11 GB total
  Missing duration:    0
  Missing recorded_at: 0



,original_filename,uuid,video_path,thumbnail_path,file_size_mb,duration,recorded_at,tags,mime_type
0,IMG_0033.MOV,5beaf6ac-1976-4bfd-ba83-1b780de45f16,videos/5beaf6ac-1976-4bfd-ba83-1b780de45f16.mov,thumbs/5beaf6ac-1976-4bfd-ba83-1b780de45f16.webp,27.8,0:26,2025-11-23T02:07:20+00:00,Bachata,video/quicktime
1,IMG_0034.MOV,80246e00-607a-46d1-873a-06ff6aba44ab,videos/80246e00-607a-46d1-873a-06ff6aba44ab.mov,thumbs/80246e00-607a-46d1-873a-06ff6aba44ab.webp,48.1,0:45,2025-11-23T02:07:51+00:00,Bachata,video/quicktime
2,IMG_0035.MOV,950e0441-e7f8-4a48-8a17-3f2342418f8e,videos/950e0441-e7f8-4a48-8a17-3f2342418f8e.mov,thumbs/950e0441-e7f8-4a48-8a17-3f2342418f8e.webp,23.9,0:22,2025-11-23T02:08:37+00:00,Bachata,video/quicktime
3,IMG_0885.MP4,aae7dbce-500c-48ea-bcee-1eee964a066f,videos/aae7dbce-500c-48ea-bcee-1eee964a066f.mp4,thumbs/aae7dbce-500c-48ea-bcee-1eee964a066f.webp,101.6,0:56,2026-01-06T03:20:58+00:00,Bachata,video/mp4
4,IMG_1105.MOV,08471a54-581b-417c-8802-4da4c1baf6a7,videos/08471a54-581b-417c-8802-4da4c1baf6a7.mov,thumbs/08471a54-581b-417c-8802-4da4c1baf6a7.webp,356.3,3:11,2026-01-20T06:00:44+00:00,Bachata,video/quicktime
5,IMG_1460.MOV,31400aa7-7002-40df-ae67-dffbf15dd27a,videos/31400aa7-7002-40df-ae67-dffbf15dd27a.mov,thumbs/31400aa7-7002-40df-ae67-dffbf15dd27a.webp,91.5,0:49,2026-02-25T06:06:24+00:00,Bachata,video/quicktime
6,IMG_1912.MP4,8cff3f7f-42a9-4a96-aa47-a26d6d1f0041,videos/8cff3f7f-42a9-4a96-aa47-a26d6d1f0041.mp4,thumbs/8cff3f7f-42a9-4a96-aa47-a26d6d1f0041.webp,61.9,0:27,2024-08-29T04:20:11+00:00,Bachata,video/mp4
7,IMG_6050.MOV,40baab27-542b-401e-98c2-a53b50634e3a,videos/40baab27-542b-401e-98c2-a53b50634e3a.mov,thumbs/40baab27-542b-401e-98c2-a53b50634e3a.webp,20.4,0:19,2025-04-09T03:16:47+00:00,Bachata,video/quicktime
8,IMG_6051.MOV,e453be2d-fc90-4b47-b034-50b675b9389f,videos/e453be2d-fc90-4b47-b034-50b675b9389f.mov,thumbs/e453be2d-fc90-4b47-b034-50b675b9389f.webp,80.5,1:16,2025-04-09T03:17:20+00:00,Bachata,video/quicktime
9,IMG_6099.MOV,1287ef8c-907a-4c54-bc3a-6f4608f367f4,videos/1287ef8c-907a-4c54-bc3a-6f4608f367f4.mov,thumbs/1287ef8c-907a-4c54-bc3a-6f4608f367f4.webp,59.3,0:56,2025-04-12T21:11:24+00:00,Bachata,video/quicktime


---
## Step 4: Upload

Executes the plan built in Step 3. Each entry already has its UUID, paths, duration, and metadata pre-computed — nothing is recalculated here.

For each video:
1. Upload video file to `videos/{uuid}.{ext}` in R2
2. Generate thumbnail via ffmpeg → upload to `thumbs/{uuid}.webp` in R2
3. Insert Supabase media record with all metadata
4. Insert `media_tags` rows

In [6]:
import subprocess
import tempfile
import imageio_ffmpeg

FFMPEG_PATH = imageio_ffmpeg.get_ffmpeg_exe()
TARGET_WIDTH = 640
WEBP_QUALITY = 80
SEEK_TIME_SEC = 1.0


def generate_thumbnail(video_path: str) -> bytes | None:
    """Extract frame at ~1s, scale to 640px wide, encode as WebP @ 80% quality."""
    with tempfile.NamedTemporaryFile(suffix=".webp", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            [
                FFMPEG_PATH, "-ss", str(SEEK_TIME_SEC), "-i", video_path,
                "-vframes", "1", "-vf", f"scale={TARGET_WIDTH}:-2",
                "-quality", str(WEBP_QUALITY), "-y", tmp_path,
            ],
            capture_output=True, timeout=60,
        )
        if result.returncode != 0:
            # Retry at frame 0 for very short videos
            result = subprocess.run(
                [
                    FFMPEG_PATH, "-i", video_path,
                    "-vframes", "1", "-vf", f"scale={TARGET_WIDTH}:-2",
                    "-quality", str(WEBP_QUALITY), "-y", tmp_path,
                ],
                capture_output=True, timeout=60,
            )
        if result.returncode != 0 or not os.path.exists(tmp_path):
            return None
        with open(tmp_path, "rb") as f:
            return f.read()
    except (subprocess.TimeoutExpired, OSError):
        return None
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


print(f"Using ffmpeg: {FFMPEG_PATH}")
print(f"Ready to upload {len(plan)} files.")
if DRY_RUN:
    print("DRY_RUN = True — set it to False in the config cell to actually upload.")

Using ffmpeg: c:\Users\minds\miniforge3\envs\Primary\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe
Ready to upload 351 files.
DRY_RUN = True — set it to False in the config cell to actually upload.


In [7]:
if DRY_RUN:
    print("DRY_RUN is True — nothing will be uploaded. Set DRY_RUN = False to proceed.")
else:
    results = {"success": 0, "failed": 0}
    errors = []

    for i, p in enumerate(plan):
        fname = p["original_filename"]
        prefix = f"[{i+1}/{len(plan)}] {fname}"

        media_key = p["video_path"]
        thumb_key = p["thumbnail_path"]
        mime_type = p["mime_type"]
        fpath = p["_filepath"]
        fsize = p["_file_size_bytes"]
        duration = p["_duration_secs"]
        tag_ids = p["_tag_ids"]

        # ── 1. Upload video to R2 ──
        print(f"{prefix} uploading... ", end="", flush=True)
        try:
            with open(fpath, "rb") as f:
                r2.put_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=media_key,
                    Body=f,
                    ContentType=mime_type,
                    ContentLength=fsize,
                )
            print("video OK.", end=" ", flush=True)
        except Exception as e:
            print(f"FAILED (video): {e}")
            errors.append((fname, f"video upload: {e}"))
            results["failed"] += 1
            continue

        # ── 2. Generate & upload thumbnail ──
        thumb_bytes = generate_thumbnail(fpath)
        actual_thumb_path = None
        if thumb_bytes:
            try:
                r2.put_object(
                    Bucket=R2_BUCKET_NAME,
                    Key=thumb_key,
                    Body=thumb_bytes,
                    ContentType="image/webp",
                )
                actual_thumb_path = thumb_key
                print("thumb OK.", end=" ", flush=True)
            except Exception as e:
                print(f"thumb upload failed ({e}).", end=" ", flush=True)
        else:
            print("thumb gen failed.", end=" ", flush=True)

        # ── 3. Create DB record ──
        title = os.path.splitext(fname)[0]
        try:
            media_res = sb.table("media").insert({
                "title":             title,
                "media_type":        "video",
                "storage_path":      media_key,
                "thumbnail_path":    actual_thumb_path,
                "original_filename": fname,
                "file_size_bytes":   fsize,
                "mime_type":         mime_type,
                "duration":          duration,
                "recorded_at":       p["recorded_at"],
                "uploaded_by":       None,  # patched in Step 5
            }).select("id").single().execute()
            media_id = media_res.data["id"]
        except Exception as e:
            print(f"FAILED (DB): {e}")
            errors.append((fname, f"DB insert: {e}"))
            results["failed"] += 1
            continue

        # ── 4. Insert media_tags ──
        if tag_ids:
            try:
                sb.table("media_tags").insert([
                    {"media_id": media_id, "tag_id": tid, "created_by": None}
                    for tid in tag_ids
                ]).execute()
            except Exception as e:
                print(f"tags failed ({e}).", end=" ", flush=True)
                errors.append((fname, f"media_tags: {e}"))

        print("record OK.")
        results["success"] += 1

    print()
    print("=" * 60)
    print("UPLOAD COMPLETE")
    print("=" * 60)
    print(f"  Uploaded successfully: {results['success']}")
    print(f"  Failed:               {results['failed']}")
    print(f"  Skipped (duplicates): {len(duplicates)}")
    if errors:
        print()
        print("Errors:")
        for fname, err in errors:
            print(f"  {fname}: {err}")

DRY_RUN is True — nothing will be uploaded. Set DRY_RUN = False to proceed.


---
## Step 5: Set `uploaded_by` (required for media_tags)

The service role key bypasses RLS so `uploaded_by` won't be set automatically. Run this cell once to patch any records just created that are missing it.

In [8]:
# ── Patch uploaded_by on records missing it ──
# Set this to your user profile ID from the profiles table
YOUR_USER_ID = "5d24bf9b-2992-4649-a99b-5649882997f1"  # <-- paste your UUID here, e.g. "a1b2c3d4-..."

if not YOUR_USER_ID:
    print("Set YOUR_USER_ID above to patch uploaded_by.")
    print()
    # Show all profile IDs to help you find yours
    profiles = sb.table("profiles").select("id, display_name, email").execute()
    print(tabulate(
        [[p["id"], p.get("display_name", ""), p.get("email", "")] for p in profiles.data],
        headers=["ID", "Display Name", "Email"],
        tablefmt="simple"
    ))
else:
    res = sb.table("media").update({"uploaded_by": YOUR_USER_ID}).is_("uploaded_by", "null").execute()
    print(f"Patched uploaded_by on {len(res.data)} records.")

    # Also fix any media_tags rows with null created_by
    sb.table("media_tags").update({"created_by": YOUR_USER_ID}).is_("created_by", "null").execute()
    print("Patched created_by on media_tags rows.")

Patched uploaded_by on 0 records.
Patched created_by on media_tags rows.
